In [ ]:
# 대회 환경 맞추기
!pip uninstall -y torch transformers -q

!pip install torch
!pip install transformers==4.57.3
!pip install tokenizers==0.22.1
!pip install accelerate==1.10.1
!pip install datasets==4.4.1
!pip install compressed-tensors==0.13.0

# llmcompressor 설치 시도
!pip install llmcompressor

print("설치 완료 - 런타임 재시작 필요!")

  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl (915.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.2 requires transformers<6.0.0,>=4.41.0, which is not installed.
peft 0.18.1 requires transformers, which is not installed.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.10.0 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2

# Import

In [2]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# Setting

In [3]:
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
OUT_DIR  = "./model"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 512 # 캘리브레이션 데이터 수 증가 256-> 512
MAX_SEQUENCE_LENGTH = 1024 # 512 -> 1024

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

# Model Loads

In [4]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

[INFO] 모델/토크나이저 로드 완료


# IGNORE LIST

In [5]:
# (추가) 모든 층 down_proj + 마지막 4개 층 보호
n_layers = model.config.num_hidden_layers
ignore_list = ["model.embed_tokens", "model.norm", "lm_head"]

for i in range(n_layers):
    # down_proj
    ignore_list.append(f"model.layers.{i}.mlp.down_proj")

    # 26.27.28.29(에러 많은 레이어)
    if i >= n_layers - 4:
        ignore_list.append(f"model.layers.{i}.self_attn.q_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.k_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.v_proj")
        ignore_list.append(f"model.layers.{i}.self_attn.o_proj")
        ignore_list.append(f"model.layers.{i}.mlp.gate_proj")
        ignore_list.append(f"model.layers.{i}.mlp.up_proj")

print(f"[DEBUG] 제외 대상(IGNORE) 설정 완료: 총 {len(ignore_list)}개 모듈 제외됨")
print(f"[INFO] Layer {n_layers-4} ~ {n_layers-1} (총 4개 층)은 완전히 보호됩니다.")

[DEBUG] 제외 대상(IGNORE) 설정 완료: 총 57개 모듈 제외됨
[INFO] Layer 26 ~ 29 (총 4개 층)은 완전히 보호됩니다.


# Dataset Loads & Preprocess

In [6]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [ ]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore_list,
        dampening_frac=0.2 # dampening_frac = 0.01 -> 0.2
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=512, max_len=1024)...


Tokenizing:   0%|          | 0/512 [00:00<?, ? examples/s]

2026-02-10T07:24:09.958407+0000 | reset | INFO - Compression lifecycle reset
2026-02-10T07:24:09.961370+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-10T07:24:10.183048+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-10T07:24:10.183904+0000 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`


(1/31): Calibrating: 100%|██████████| 512/512 [00:06<00:00, 83.84it/s]

2026-02-10T07:24:17.397191+0000 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 512 samples


...

2026-02-10T07:26:15.522506+0000 | compress_modules | INFO - Quantizing model.layers.7.self_attn.q_proj using 512 samples


2026-02-10T07:26:16.655443+0000 | compress | METRIC - time 1.13s
2026-02-10T07:26:16.657248+0000 | compress | METRIC - error 449.62
2026-02-10T07:26:16.658091+0000 | compress | METRIC - GPU 0 | usage: 3.97% | total memory: 24 GB
2026-02-10T07:26:16.658664+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-10T07:26:16.659447+0000 | compress_modules | INFO - Quantizing model.layers.7.self_attn.k_proj using 512 samples
2026-02-10T07:26:17.781145+0000 | compress | METRIC - time 1.12s
2026-02-10T07:26:17.782910+0000 | compress | METRIC - error 126.58
2026-02-10T07:26:17.783752+0000 | compress | METRIC - GPU 0 | usage: 3.97% | total memory: 24 GB
2026-02-10T07:26:17.784337+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-10T07:26:17.785362+0000 | compress_modules | INFO - Quantizing model.layers.7.self_attn.v_proj using 512 samples
2026-02-10T07:26:18.904840+0000 | compress | METRIC - time 1.12s
2026-02-10T07:26:18.906807+0000 | compress | METRIC - e

(9/31): Calibrating: 100%|██████████| 512/512 [00:05<00:00, 94.07it/s]

2026-02-10T07:26:30.431212+0000 | compress_modules | INFO - Quantizing model.layers.8.self_attn.q_proj using 512 samples


2026-02-10T07:26:31.581755+0000 | compress | METRIC - time 1.15s
2026-02-10T07:26:31.583516+0000 | compress | METRIC - error 504.59
2026-02-10T07:26:31.584272+0000 | compress | METRIC - GPU 0 | usage: 3.97% | total memory: 24 GB
2026-02-10T07:26:31.584798+0000 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-10T07:26:31.585932+0000 | compress_modules | INFO - Quantizing model.layers.8.self_attn.k_proj using 512 samples
2026-02-10T07:26:32.713353+0000 | compress | METRIC - time 1.13s
2026-02-10T07:26:32.715204+0000 | compress | METRIC - error 144.84
2026-02-10T07:26:32.715927+0000 | compress | METRIC - GPU 0 | usage: 3.97% | total memory: 24 GB
2026-02-10T07:26:32.716433+0000 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-10T07:26:32.717323+0000 | compress_modules | INFO - Quantizing model.layers.8.self_attn.v_proj using 512 samples
2026-02-10T07:26:33.832553+0000 | compress | METRIC - time 1.11s
2026-02-10T07:26:33.834470+0000 | compress | METRIC - e

# Model Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-09T07:13:03.236080+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 105.47it/s]


[INFO] 모델 저장 완료: ./model


# Submission

In [ ]:
zip_name = "GPTQ_mcl_2084"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] GPTQ_baseline_cpu.zip 생성 중...
[INFO] 생성 완료: GPTQ_baseline_cpu.zip


In [ ]:
from google.colab import files
files.download("GPTQ_mcl_2084.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# lm-evaluation-harness 테스트



In [ ]:
!pip uninstall -y lm_eval lm-eval

Found existing installation: lm_eval 0.4.10
Uninstalling lm_eval-0.4.10:
  Successfully uninstalled lm_eval-0.4.10


In [ ]:
!pip install -q lm-eval[vllm]

In [ ]:
!rm -rf /content/eval_model
!mkdir -p /content/eval_model
!unzip -o "GPTQ_baseline_cpu.zip" -d "/content/eval_model" > /dev/null


unzip:  cannot find or open GPTQ_baseline_cpu.zip, GPTQ_baseline_cpu.zip.zip or GPTQ_baseline_cpu.zip.ZIP.


In [ ]:
!find /content/eval_model -maxdepth 3 -name "config.json"

In [ ]:
model_dir = "/content/eval_model/model"

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

In [ ]:
!python -m lm_eval --model vllm \
  --model_args pretrained=/content/eval_model/model,gpu_memory_utilization=0.8 \
  --tasks gsm8k \
  --device cuda \
  --batch_size auto


2026-02-10:02:42:37 INFO     [_cli.run:376] Selected Tasks: ['gsm8k']
2026-02-10:02:42:39 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-10:02:42:39 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': '/content/eval_model/model', 'gpu_memory_utilization': 0.8}
2026-02-10 02:42:45.728234: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770691365.752756   20099 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770691365.759999   20099 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770691365.777284   2